In [174]:
import uuid
from urllib.parse import urlparse

class utils:
    """
    Utility class for handling endpoint strings and logging properties.

    Methods:
        slashCheck(endpoint): Removes leading slash if present.
        slashConverter(endpoint): Converts forward slashes to backslashes.
        createUuid(): Generates a unique UUID string.
        joinPath(endpoint, separator): Joins last 3 parts of a path with a separator.
        disablePayloadProp(endpoint, separator): Builds a log property string.
    """

    ##############################################
    #########                            #########
    ######     Common Utility Methods      ######
    #########                            #########
    ##############################################

    # Slash Check: Remove leading slash if present
    def slashCheck(self, endpoint: str) -> str:
        """
        Remove leading slash from the endpoint if it exists.

        Parameters:
            endpoint (str): The API endpoint string (e.g., "/a/b/c").

        Returns:
            str: Endpoint without leading slash.
        """
        if(endpoint.startswith("/")):
            return endpoint[1:]
        else:
            return endpoint
    
    # Slash Converter: Convert forward slashes to backslashes
    def slashConverter(self, endpoint: str) -> str:
        """
        Convert forward slashes to backslashes in the endpoint.

        Parameters:
            endpoint (str): The API endpoint string (e.g., "/a/b/c").

        Returns:
            str: Endpoint with backslashes (e.g., "\a\b\c").
        """
        return endpoint.replace("/", "\\")

    # UUID Generation: Generate a unique identifier
    def createUuid(self) -> str:
        """
        Generate a random UUID string.

        Returns:
            str: A unique identifier (e.g., "123e4567-e89b-12d3-a456-426614174000").
        """
        return str(uuid.uuid4())

    # Join Path: Join last 3 parts of the endpoint with a separator
    def joinPath(self, endpoint: str, separator: str = ".") -> str:
        """
        Join the last 3 parts of the endpoint path using a separator.

        Parameters:
            endpoint (str): The API endpoint string (e.g., "/a/b/c/d/e").
            separator (str): The character to join with (default is ".").

        Returns:
            str: Joined path (e.g., "c.d.e").
        """
        endpointSplit= [p for p in endpoint.split("/")[-3:] if p] 
        return separator.join(endpointSplit)
    
    # Disable Payload Property: Build a log property string for disabling payload logs
    def disablePayloadProp(self, endpoint: str, separator: str = ".") -> str:
        """
        Build a log property string for disabling payload logs.

        Parameters:
            endpoint (str): The API endpoint string.
            separator (str): The character to join with (default is ".").

        Returns:
            str: Log property string (e.g., "a.f.disable.payload.logs").
        """
        result = self.joinPath(endpoint, separator)
        return result + ".disable.payload.logs"

    # Get Backend Details
    def getBackendDetails(self, backendURL, withType):
        if(withType == "request"):
            return urlparse(backendURL)
        else:
            raise ValueError("Define proper backend type")
            
    # Get CallingFunction
    def sysLoggerMessage(self, backendURL, withType):
        if (withType == "request"):
            return f"{urlparse(backendURL).path}"
        else:
            raise ValueError("Define proper backend type")
        
        
    ##############################################
    #########                            #########
    ###### Flow Name & File Name Generation ######
    #########                            #########
    ##############################################

    # Implementation Flow Name: Generate a standardized implementation flow name
    def implFlowName(self, endpoint: str) -> str:
        """
        Generate a standardized implementation flow name for MuleSoft subflows.

        Parameters:
            endpoint (str): The API endpoint string (e.g., "/a/b/c/d/e").

        Returns:
            str: Flow name in the format "impl-x-y-z-subflow",
                 where x-y-z are the last three parts of the endpoint joined by "-".

        Example:
            >>> utilsObj = Utils()
            >>> utilsObj.implFlowName("/a/b/c/d/e")
            'impl-c-d-e-subflow'
        """
        return f"impl-{self.joinPath(endpoint, "-")}-subflow"
    
    # Implementation File Name: Generate a standardized implementation file name
    def implFileName(self, endpoint: str) -> str:
        """
        Generate a standardized implementation file name for MuleSoft subflows.

        Parameters:
            endpoint (str): The API endpoint string (e.g., "/a/b/c/d/e").

        Returns:
            str: Flow name in the format "impl-x-y-z-subflow",
                 where x-y-z are the last three parts of the endpoint joined by "-".

        Example:
            >>> utilsObj = Utils()
            >>> utilsObj.implFileName("/a/b/c/d/e")
            'impl-c-d-e-flows'
        """
        return f"impl-{self.joinPath(endpoint, "-")}-flows.xml"
                
    # System File Name: Generate a standardized system file name
    def sysFileName(self, backendUrl: str, whichType: str) -> str:
        return f"{self.getBackendDetails(backendUrl, whichType).hostname}-flows.xml"

    # System Flow Name: Generate a standardized system flow name
    def sysFlowName(self,backendUrl: str , endpoint: str, whichType: str) -> str:
        return f"{self.getBackendDetails(backendUrl, whichType).hostname}-api-{self.joinPath(endpoint, "-")}-subflow"
    

    ################################################
    #########                              #########
    ######      Connector Preparation         ######
    #########                              #########
    ################################################

    def prepareHttpConnector(self, backendUrl: str,endpoint: str, method: str = "POST") -> str:
        """
        Prepare the HTTP connector configuration for MuleSoft.

        Parameters:
            backendUrl (str): The backend URL (e.g., "http://backend-service/api").
            endpoint (str): The API endpoint path (e.g., "/a/b/c").
            method (str): The HTTP method (default is "POST").

        Returns:
            str: A formatted MuleSoft HTTP connector configuration string.

        Example:
            >>> utilsObj = Utils()
            >>> config = utilsObj.prepareHttpConnector("http://backend-service/api")
            >>> print(config)
            <http:request-config name="http-request-config" ...>
                ...
            </http:request-config>
        """
        http_connector_config = f"""
        <http:request method="${method}" doc:name="Request" doc:id="{self.createUuid()}" config-ref="HTTP_Request_configuration_{self.getBackendDetails(backendUrl, 'request').hostname}" path='${{{self.backendPropName(backendUrl, 'request', endpoint).get('path')}}}' >
			<http:headers ><![CDATA[#[%dw 2.0
output application/json
---
{{}}]]]></http:headers>
		</http:request>
        """
        return http_connector_config
    
    ################################################
    #########                              #########
    ######      property name generation      ######
    #########                              #########
    ################################################

    def backendPropName(self, backendUrl: str, whichType: str, endpoint: str) -> dict:
        """
        Generate a standardized property name for backend configurations.

        Parameters:
            backendUrl (str): The backend URL (e.g., "http://backend-service/api").
            whichType (str): The type of property (e.g., "request").
            endpoint (str): The API endpoint path (e.g., "/a/b/c").

        Returns:
            dict: A dictionary of standardized property names.

        Example:
            >>> utilsObj = Utils()
            >>> propName = utilsObj.backendPropName("http://backend-service/api", "request", "/a/b/c")
            >>> print(propName)
            'backend-service-request-config'
        """
        result  = {
            "protocol": f"{self.getBackendDetails(backendUrl, whichType).hostname}.protocol",
            "hostname": f"{self.getBackendDetails(backendUrl, whichType).hostname}.hostname",
            "port": f"{self.getBackendDetails(backendUrl, whichType).hostname}.port",
            "basePath": f"{self.getBackendDetails(backendUrl, whichType).hostname}.basePath",
            "path": f"{self.getBackendDetails(backendUrl, whichType).hostname}.{self.joinPath(endpoint, '.')}.path",
            "connectionTimeout": f"{self.getBackendDetails(backendUrl, whichType).hostname}.connectionTimeout",
            "responseTimeout": f"{self.getBackendDetails(backendUrl, whichType).hostname}.responseTimeout",
            "usePersistentConnections": f"{self.getBackendDetails(backendUrl, whichType).hostname}.usePersistentConnections",
            "insecure": f"{self.getBackendDetails(backendUrl, whichType).hostname}.insecure",
        }
        return result
    


    ################################################
    #########                              #########
    ######      Default System File      ######
    #########                              #########
    ################################################

    def defaultSystemFile(self, whichType: str, flowCode: str) -> str:
        if whichType == "request":
            result = f"""
<?xml version="1.0" encoding="UTF-8"?>
<mule xmlns:http="http://www.mulesoft.org/schema/mule/http" 
    xmlns:ee="http://www.mulesoft.org/schema/mule/ee/core" 
    xmlns="http://www.mulesoft.org/schema/mule/core" 
    xmlns:doc="http://www.mulesoft.org/schema/mule/documentation" 
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.mulesoft.org/schema/mule/core http://www.mulesoft.org/schema/mule/core/current/mule.xsd
http://www.mulesoft.org/schema/mule/ee/core http://www.mulesoft.org/schema/mule/ee/core/current/mule-ee.xsd
http://www.mulesoft.org/schema/mule/http http://www.mulesoft.org/schema/mule/http/current/mule-http.xsd">
{flowCode}
</mule>
            """
            return result
    def createRequestConfig(self, backendUrl: str,endpoint : str , backendType:str = "request") -> str:
        configName = f"HTTP_Request_configuration_{self.getBackendDetails(backendUrl, backendType).hostname}-config"
        xmlCode = f"""<http:request-config name="{configName}" doc:name="{configName}" doc:id="bc0d1f13-0adb-4527-a826-b5846877ea92" basePath="${{{self.backendPropName(backendUrl, backendType, endpoint)["basePath"]}}}" responseTimeout="${{{self.backendPropName(backendUrl, backendType, endpoint)["responseTimeout"]}}}" >
		<http:request-connection protocol="${{{self.backendPropName(backendUrl, backendType, endpoint)["protocol"]}}}" host="${{{self.backendPropName(backendUrl, backendType, endpoint)["hostname"]}}}" port="${{{self.backendPropName(backendUrl, backendType, endpoint)["port"]}}}" usePersistentConnections="${{{self.backendPropName(backendUrl, backendType, endpoint)["usePersistentConnections"]}}}" connectionIdleTimeout="${{{self.backendPropName(backendUrl, backendType, endpoint)["connectionTimeout"]}}}" >
			<tls:context >
				<tls:trust-store insecure="${{{self.backendPropName(backendUrl, backendType, endpoint)["insecure"]}}}" />
			</tls:context>
		</http:request-connection>
	</http:request-config>
        """
        return xmlCode

<>:47: SyntaxWarning: invalid escape sequence '\c'
<>:47: SyntaxWarning: invalid escape sequence '\c'
C:\Users\aadhi\AppData\Local\Temp\ipykernel_19424\1952925388.py:47: SyntaxWarning: invalid escape sequence '\c'
  str: Endpoint with backslashes (e.g., "\a\b\c").


In [175]:
utilsObj = utils()

In [177]:
print(utilsObj.createRequestConfig("http://sys-mobile-api/test","demo/test"))

<http:request-config name="HTTP_Request_configuration_sys-mobile-api-request-config" doc:name="HTTP_Request_configuration_sys-mobile-api-request-config" doc:id="bc0d1f13-0adb-4527-a826-b5846877ea92" basePath="${sys-mobile-api.basePath}" responseTimeout="${sys-mobile-api.responseTimeout}" >
		<http:request-connection protocol="${sys-mobile-api.protocol}" host="${sys-mobile-api.hostname}" port="${sys-mobile-api.port}" usePersistentConnections="${sys-mobile-api.usePersistentConnections}" connectionIdleTimeout="${sys-mobile-api.connectionTimeout}" >
			<tls:context >
				<tls:trust-store insecure="${sys-mobile-api.insecure}" />
			</tls:context>
		</http:request-connection>
	</http:request-config>
        


In [105]:
print(utilsObj.prepareHttpConnector( "http://sys-mobile-api/test", "/services/test"))



        <http:request method="$POST" doc:name="Request" doc:id="9ec271c7-99ea-45ef-95fb-0cf2c4e030f1" config-ref="HTTP_Request_configuration_sys-mobile-api" path='${sys-mobile-api.services.test.path}' >
			<http:headers ><![CDATA[#[%dw 2.0
output application/json
---
{}]]]></http:headers>
		</http:request>
        


In [153]:
def createRequestConfig(self, backendUrl: str) -> str:
        result = f"{self.getBackendDetails(backendUrl, 'request').hostname}-request-config"
        return result

In [160]:
utilsObj.backendPropName("http://sys-mobile-api/test", "request", "/demo/test")

{'protocol': 'sys-mobile-api.protocol',
 'hostname': 'sys-mobile-api.hostname',
 'port': 'sys-mobile-api.port',
 'basePath': 'sys-mobile-api.basePath',
 'path': 'sys-mobile-api.demo.test.path',
 'connectionTimeout': 'sys-mobile-api.connectionTimeout',
 'responseTimeout': 'sys-mobile-api.responseTimeout',
 'usePersistentConnections': 'sys-mobile-api.usePersistentConnections',
 'insecure': 'sys-mobile-api.insecure'}